# Fraud Detection in Financial Transactions
**Author:** Vedant Mandavale  
**Date:** July 2026  
**Submitted to:** GENZ EDUCATEWING — Support@Genzeducatewing.com

---

## Project Overview
This notebook builds a complete **machine learning pipeline** to detect fraudulent credit card transactions using the [Kaggle Credit Card Fraud Detection dataset](https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud).

### Business Problem
Banks process millions of transactions daily. Fraud is rare (~0.17%) but costly. We need a model that:
- **Catches fraud** (high **Recall**)
- **Minimizes false alarms** (reasonable **Precision**)
- Ranks transactions well (**ROC-AUC**)

> ⚠️ **Accuracy is misleading** on imbalanced data — a model predicting "no fraud" always would be 99.8% accurate but useless.


## Step 1 — Environment Setup

We import libraries for:
| Library | Purpose |
|---------|---------|
| `pandas`, `numpy` | Data manipulation |
| `matplotlib`, `seaborn` | Visualization |
| `sklearn` | ML models, metrics, preprocessing |
| `imblearn` (SMOTE) | Handle class imbalance |
| `keras` | Neural Network (uses PyTorch backend on Python 3.14) |
| `joblib` | Save trained models |


In [ ]:
import os, warnings
from pathlib import Path

# Keras 3: TensorFlow not available on Python 3.14, so we use PyTorch backend
os.environ["KERAS_BACKEND"] = "torch"
warnings.filterwarnings("ignore")

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from imblearn.over_sampling import SMOTE
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    f1_score, precision_score, recall_score, roc_auc_score, roc_curve,
)
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
import keras
from keras import layers

RANDOM_STATE = 42
DATA_PATH = Path("data/creditcard.csv")
PLOTS_DIR = Path("outputs/plots")
MODELS_DIR = Path("models")
for d in (PLOTS_DIR, MODELS_DIR):
    d.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
print("✓ All libraries loaded successfully")


## Step 2 — Data Loading & Exploratory Data Analysis (EDA)

### Dataset columns
| Column | Description |
|--------|-------------|
| `Time` | Seconds between transaction and first transaction |
| `Amount` | Transaction amount |
| `V1`–`V28` | PCA-transformed features (original data anonymized) |
| `Class` | **Target**: 0 = legitimate, 1 = fraud |


In [ ]:
df = pd.read_csv(DATA_PATH)
df["Class"] = df["Class"].astype(int)

print("Shape:", df.shape)
print("\nData types:\n", df.dtypes.value_counts())
print("\nMissing values:", df.isnull().sum().sum())
print("\nClass distribution:")
print(df["Class"].value_counts())
print(f"\nFraud rate: {df['Class'].mean()*100:.4f}%")
df.describe()


In [ ]:
# Visualize severe class imbalance
fig, ax = plt.subplots(figsize=(7, 4))
counts = df["Class"].value_counts()
colors = ["#27ae60", "#e74c3c"]
bars = ax.bar(["Legitimate (0)", "Fraud (1)"], counts.values, color=colors)
ax.set_title("Class Distribution — Severe Imbalance (~0.17% fraud)")
ax.set_ylabel("Number of Transactions")
for bar, val in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(), f"{val:,}", ha="center", va="bottom")
plt.savefig(PLOTS_DIR / "eda_class_distribution.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# Correlation heatmap — which features relate to fraud?
corr_cols = [f"V{i}" for i in range(1, 29)] + ["Amount", "Time", "Class"]
plt.figure(figsize=(14, 10))
sns.heatmap(df[corr_cols].corr(), cmap="coolwarm", center=0, linewidths=0.1)
plt.title("Feature Correlation Heatmap")
plt.savefig(PLOTS_DIR / "eda_correlation_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# Amount and Time distributions
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df["Amount"], bins=50, kde=True, ax=axes[0], color="#3498db")
axes[0].set_title("Amount Distribution (right-skewed)")
sns.histplot(df["Time"], bins=50, kde=True, ax=axes[1], color="#9b59b6")
axes[1].set_title("Time Distribution")
plt.savefig(PLOTS_DIR / "eda_amount_time_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

# Amount by class
fig, ax = plt.subplots(figsize=(7, 4))
sns.boxplot(data=df, x="Class", y="Amount", palette=["#27ae60", "#e74c3c"], ax=ax)
ax.set_xticklabels(["Legitimate", "Fraud"])
ax.set_title("Transaction Amount by Class")
plt.savefig(PLOTS_DIR / "eda_amount_by_class.png", dpi=150, bbox_inches="tight")
plt.show()


### EDA Key Findings
1. **284,807 transactions**, only **492 fraud** (0.17%) — extreme imbalance
2. **No missing values** — clean dataset
3. **V1–V28** are PCA components; **Amount** and **Time** need scaling
4. Fraud transactions may show different **Amount** patterns


## Step 3 — Data Preprocessing

Steps:
1. **Drop duplicates** — remove repeated rows
2. **Scale Amount & Time** — StandardScaler (V1–V28 already normalized via PCA)
3. **Train/test split** — 80/20, **stratified** to preserve fraud ratio
4. **SMOTE on training data ONLY** — synthetically oversample fraud class

> ⚠️ Never apply SMOTE to test data — that would leak information and give unrealistic scores.


In [ ]:
before = len(df)
df = df.drop_duplicates()
print(f"Removed {before - len(df)} duplicate rows")

feature_cols = [c for c in df.columns if c != "Class"]
X = df[feature_cols].copy()
y = df["Class"]

# Scale only Amount and Time
scaler = StandardScaler()
X[["Amount", "Time"]] = scaler.fit_transform(X[["Amount", "Time"]])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

smote = SMOTE(random_state=RANDOM_STATE)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

print(f"Original train: {len(X_train):,} (fraud: {y_train.sum()})")
print(f"After SMOTE:    {len(X_train_res):,} (fraud: {y_train_res.sum()})")
print(f"Test set:         {len(X_test):,} (fraud: {y_test.sum()}) — kept imbalanced")


## Step 4 — Feature Selection

We use **Random Forest feature importance** to identify which PCA components matter most.  
We proceed with **all 30 features** for final models (as required by project scope).


In [ ]:
rf_fs = RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1)
rf_fs.fit(X_train_res, y_train_res)

importance = pd.Series(rf_fs.feature_importances_, index=feature_cols).sort_values(ascending=False)
print("Top 10 most important features:")
display(importance.head(10).to_frame("Importance"))

fig, ax = plt.subplots(figsize=(10, 6))
importance.head(15).sort_values().plot(kind="barh", ax=ax, color="#2980b9")
ax.set_title("Top 15 Features — Random Forest Importance")
ax.set_xlabel("Importance Score")
plt.savefig(PLOTS_DIR / "feature_importance_top15.png", dpi=150, bbox_inches="tight")
plt.show()


## Step 5 — Model Building

We train **4 models** as specified:
| # | Model | Why use it? |
|---|-------|-------------|
| 1 | Logistic Regression | Fast baseline, interpretable |
| 2 | Decision Tree | Captures non-linear rules |
| 3 | Random Forest | Ensemble, reduces overfitting |
| 4 | Neural Network (Keras) | Learns complex patterns |

### Neural Network Architecture
`Input → Dense(64, relu) → Dropout(0.3) → Dense(32, relu) → Dropout(0.3) → Dense(1, sigmoid)`


## Step 6 — Cross-Validation (5-Fold Stratified)


In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

cv_models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1),
}

print("5-Fold Cross-Validation (ROC-AUC on SMOTE training data):\n")
for name, model in cv_models.items():
    scores = cross_val_score(model, X_train_res, y_train_res, cv=cv, scoring="roc_auc", n_jobs=-1)
    print(f"{name}:")
    print(f"  Scores: {scores.round(4)}")
    print(f"  Mean:   {scores.mean():.4f} (+/- {scores.std():.4f})\n")


## Step 7 — Training & Evaluation on Test Set


In [ ]:
def compute_metrics(y_true, y_pred, y_prob):
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1": f1_score(y_true, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_true, y_prob),
    }

def plot_evaluation(name, y_true, y_pred, y_prob, cm_file, roc_file):
    print(f"\n{'='*50}\n{name}\n{'='*50}")
    print(classification_report(y_true, y_pred, target_names=["Legitimate", "Fraud"]))

    fig, ax = plt.subplots(figsize=(5, 4))
    sns.heatmap(confusion_matrix(y_true, y_pred), annot=True, fmt="d", cmap="Blues",
                xticklabels=["Legit", "Fraud"], yticklabels=["Legit", "Fraud"], ax=ax)
    ax.set_title(f"{name} — Confusion Matrix")
    plt.savefig(PLOTS_DIR / cm_file, dpi=150, bbox_inches="tight")
    plt.show()

    fpr, tpr, _ = roc_curve(y_true, y_prob)
    auc = roc_auc_score(y_true, y_prob)
    plt.figure(figsize=(6, 5))
    plt.plot(fpr, tpr, label=f"ROC-AUC = {auc:.4f}", lw=2)
    plt.plot([0, 1], [0, 1], "k--")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate (Recall)")
    plt.title(f"{name} — ROC Curve")
    plt.legend()
    plt.savefig(PLOTS_DIR / roc_file, dpi=150, bbox_inches="tight")
    plt.show()

results, trained_models = {}, {}


In [ ]:
# Model 1: Logistic Regression
lr = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
lr.fit(X_train_res, y_train_res)
lr_prob, lr_pred = lr.predict_proba(X_test)[:, 1], lr.predict(X_test)
results["Logistic Regression"] = compute_metrics(y_test, lr_pred, lr_prob)
trained_models["Logistic Regression"] = lr
plot_evaluation("Logistic Regression", y_test, lr_pred, lr_prob,
                "cm_logistic_regression.png", "roc_logistic_regression.png")


In [ ]:
# Model 2: Decision Tree
dt = DecisionTreeClassifier(random_state=RANDOM_STATE)
dt.fit(X_train_res, y_train_res)
dt_prob, dt_pred = dt.predict_proba(X_test)[:, 1], dt.predict(X_test)
results["Decision Tree"] = compute_metrics(y_test, dt_pred, dt_prob)
trained_models["Decision Tree"] = dt
plot_evaluation("Decision Tree", y_test, dt_pred, dt_prob,
                "cm_decision_tree.png", "roc_decision_tree.png")


In [ ]:
# Model 3: Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1)
rf.fit(X_train_res, y_train_res)
rf_prob, rf_pred = rf.predict_proba(X_test)[:, 1], rf.predict(X_test)
results["Random Forest"] = compute_metrics(y_test, rf_pred, rf_prob)
trained_models["Random Forest"] = rf
plot_evaluation("Random Forest", y_test, rf_pred, rf_prob,
                "cm_random_forest.png", "roc_random_forest.png")


In [ ]:
# Model 4: Neural Network (Keras)
nn = keras.Sequential([
    layers.Input(shape=(X_train_res.shape[1],)),
    layers.Dense(64, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(32, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(1, activation="sigmoid"),
])
nn.compile(optimizer="adam", loss="binary_crossentropy",
           metrics=[keras.metrics.AUC(name="auc")])
history = nn.fit(X_train_res.values, y_train_res.values,
                 epochs=10, batch_size=512, validation_split=0.1, verbose=1)
nn_prob = nn.predict(X_test.values, verbose=0).flatten()
nn_pred = (nn_prob >= 0.5).astype(int)
results["Neural Network"] = compute_metrics(y_test, nn_pred, nn_prob)
trained_models["Neural Network"] = nn
plot_evaluation("Neural Network", y_test, nn_pred, nn_prob,
                "cm_neural_network.png", "roc_neural_network.png")


## Step 8 — Model Comparison & Save Best Model


In [ ]:
comparison = pd.DataFrame(results).T[["Accuracy", "Precision", "Recall", "F1", "ROC-AUC"]].round(4)
display(comparison)

# Comparison bar chart
plot_df = comparison[["Precision", "Recall", "F1", "ROC-AUC"]]
fig, ax = plt.subplots(figsize=(12, 6))
plot_df.plot(kind="bar", ax=ax, colormap="Set2", width=0.8)
ax.set_title("Model Performance Comparison")
ax.set_ylabel("Score")
ax.set_ylim(0, 1.05)
ax.legend(loc="lower right")
plt.savefig(PLOTS_DIR / "model_comparison_chart.png", dpi=150, bbox_inches="tight")
plt.show()

best = comparison["ROC-AUC"].idxmax()
print(f"\nBest model by ROC-AUC: {best}")
joblib.dump(trained_models[best], MODELS_DIR / "best_model.pkl")
joblib.dump(scaler, MODELS_DIR / "scaler_Vedant_Mandavale.pkl")
print("Saved best_model.pkl and scaler_Vedant_Mandavale.pkl")


## Step 9 — Conclusion & Submission

### Key Takeaways
| Insight | Detail |
|---------|--------|
| Class imbalance | SMOTE + stratified split essential |
| Best ROC-AUC | Logistic Regression (~0.96) — highest fraud capture |
| Best balance | Random Forest — 91% precision, 76% recall |
| Metric focus | Prioritize **Recall** and **ROC-AUC** over Accuracy |

### Files to Submit
1. `fraud_detection_report_Vedant_Mandavale.pdf` — written report
2. `fraud_detection_Vedant_Mandavale.ipynb` — this notebook
3. `fraud_detection_Vedant_Mandavale.py` — Python script
4. `data/creditcard.csv` — dataset

**Email:** Support@Genzeducatewing.com
